# Chatterbox TTS on Colab GPU (remote backend)

Run this notebook on Google Colab with a **GPU runtime** (T4 is free; A100 is faster). It boots the same `travisvn/chatterbox-tts-api` server you have locally, exposes it through a public Cloudflare tunnel, and prints the URL to set as `CHATTERBOX_API_URL` on your Mac.

## Setup checklist
1. **Runtime → Change runtime type → GPU** before running cell 1.
2. Run all cells. Cell 4 prints a `https://....trycloudflare.com` URL.
3. On your Mac, in the repo root, set the URL and recreate the api container:
   ```bash
   echo 'CHATTERBOX_API_URL=https://<the-trycloudflare-url>' >> .env
   docker compose --profile cpu up -d --no-deps --force-recreate api
   ```
4. (Optional) Stop the local `chatterbox-cpu` container — it's no longer needed:
   ```bash
   docker compose --profile cpu stop chatterbox-cpu
   ```
5. Trigger TTS from the frontend. Each segment will now run on Colab's GPU.

**Important:** Colab kills idle GPU sessions after ~90 minutes. Keep this tab open and check back during long jobs.

## 1. Verify GPU

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), (
    'CUDA is not available — Runtime → Change runtime type → GPU, then re-run this cell.'
)
print(f'CUDA available: {torch.cuda.is_available()}, device: {torch.cuda.get_device_name(0)}')


## 2. Install chatterbox-tts-api dependencies

The travisvn server runs from a small Python project. Cloning + installing is faster than running the Docker image inside Colab.

In [ ]:
!git clone --depth 1 https://github.com/travisvn/chatterbox-tts-api.git /content/chatterbox-tts-api

# Colab ships Python 3.12, but chatterbox-tts pins numpy<1.26 which has no
# 3.12 wheels. Provision a separate Python 3.11 venv just for the chatterbox
# subprocess; the kernel itself can stay on 3.12.
!pip install -q uv
!uv venv --python 3.11 /content/venv311
!/content/venv311/bin/python -m pip install --upgrade pip wheel

# Install the chatterbox model package (git source) into the 3.11 venv first.
# Errors are visible (no --quiet).
!/content/venv311/bin/pip install git+https://github.com/travisvn/chatterbox-multilingual.git@exp

# Then pull in the rest of chatterbox-tts-api's runtime dependencies into
# the same venv. uvicorn[standard] is added explicitly because the
# requirements.txt extras spec sometimes drops it in fresh venvs.
!/content/venv311/bin/pip install -r /content/chatterbox-tts-api/requirements.txt
!/content/venv311/bin/pip install "uvicorn[standard]>=0.24.0" "fastapi>=0.104.0"

# Comprehensive smoke test — fail loudly here if any of the imports
# main.py needs are missing, instead of letting cell 4 surface them later.
!/content/venv311/bin/python -c "import chatterbox, uvicorn, fastapi, torch; print('all imports ok; torch=', torch.__version__, 'cuda=', torch.cuda.is_available())"


## 3. Install cloudflared (free public tunnel, no signup)

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version

## 4. Boot Chatterbox + tunnel

Both run in the background. The cell waits for the tunnel URL and prints it.

In [ ]:
import os, re, subprocess, time, pathlib

os.environ['DEVICE'] = 'cuda'
os.environ['DEFAULT_MODEL'] = 'multilingual'
os.environ['PORT'] = '8020'
os.environ['HOST'] = '0.0.0.0'

log_dir = pathlib.Path('/content/logs'); log_dir.mkdir(exist_ok=True)
tts_log_path = log_dir / 'tts.log'
tunnel_log_path = log_dir / 'tunnel.log'
tts_log = open(tts_log_path, 'wb')
tunnel_log = open(tunnel_log_path, 'w+')

# Use the Python 3.11 venv created in cell 2 — chatterbox-tts won't install
# on Colab's 3.12 due to a pinned numpy<1.26.
VENV_PYTHON = '/content/venv311/bin/python'

tts = subprocess.Popen(
    [VENV_PYTHON, 'main.py'],
    cwd='/content/chatterbox-tts-api',
    env=os.environ.copy(),
    stdout=tts_log, stderr=subprocess.STDOUT,
)

import urllib.request, urllib.error
deadline = time.time() + 600
healthy = False
while time.time() < deadline:
    # Fail fast: if the subprocess died, surface the error immediately
    # instead of waiting the full 10-minute timeout.
    rc = tts.poll()
    if rc is not None:
        tail = pathlib.Path(tts_log_path).read_text(errors='replace').splitlines()[-50:]
        raise RuntimeError(
            f'chatterbox-tts-api exited with code {rc} before becoming healthy.\n'
            f'Last 50 lines of {tts_log_path}:\n' + '\n'.join(tail)
        )
    try:
        urllib.request.urlopen('http://127.0.0.1:8020/health', timeout=2)
        print('chatterbox-tts-api is healthy')
        healthy = True
        break
    except (urllib.error.URLError, ConnectionError):
        time.sleep(5)

if not healthy:
    raise RuntimeError(
        f'chatterbox-tts-api never became healthy after 10 min; '
        f'check {tts_log_path}'
    )

tunnel = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:8020', '--no-autoupdate'],
    stdout=tunnel_log, stderr=subprocess.STDOUT,
)

url = None
deadline = time.time() + 60
while time.time() < deadline and url is None:
    tunnel_log.flush()
    text = pathlib.Path(tunnel_log_path).read_text()
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', text)
    if match:
        url = match.group(0)
        break
    time.sleep(2)

if not url:
    raise RuntimeError(f'cloudflared never printed a URL; check {tunnel_log_path}')

print('=' * 70)
print('Public URL:', url)
print('=' * 70)
print('On your Mac:')
print(f"  echo 'CHATTERBOX_API_URL={url}' >> .env")
print('  docker compose --profile cpu up -d --no-deps --force-recreate api')


## 5. Smoke test

Confirm the public URL actually answers TTS calls before pointing the Mac at it.

In [ ]:
import urllib.request, json
req = urllib.request.Request(
    f'{url}/v1/audio/speech',
    data=json.dumps({'input': 'hola desde colab', 'response_format': 'wav'}).encode(),
    headers={'Content-Type': 'application/json'},
    method='POST',
)
with urllib.request.urlopen(req, timeout=120) as r:
    data = r.read()
print(f'WAV bytes: {len(data)}')

## 6. Keep alive

Run this last cell to block the kernel and stream the chatterbox log. As long as it's running, the tunnel stays up. Stop it with the square button when done.

In [ ]:
!tail -F /content/logs/tts.log